In [2]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
# ==========================================
# 1. Dataset-ul Adaptiv (Caută singur extensia)
# ==========================================
class CombinatEyePACSDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        # Folosim direct un DataFrame (nu un drum spre CSV) pentru a face split-ul Train/Val ușor
        self.dataframe = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Citim numele imaginii (care poate veni fie ca 10_left, fie ca 0005cfc8afb6)
        img_name = str(self.dataframe.iloc[idx, 0])
        
        # Deoarece folderele "resized_" uneori schimbă .jpeg în .png, facem un căutător inteligent
        img_path = os.path.join(self.img_dir, f"{img_name}.jpeg")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir, f"{img_name}.png")
            if not os.path.exists(img_path):
                 img_path = os.path.join(self.img_dir, f"{img_name}.jpg")
        
        # Încărcare imagine
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Dacă chiar lipsește o poză din folder, returnăm un tensor negru ca să nu crape antrenamentul
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        # Extragem eticheta (numele coloanei este "level")
        label_initial = int(self.dataframe.iloc[idx, 1])
        label_binar = 0 if label_initial == 0 else 1
        label = torch.tensor(label_binar, dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [4]:
# ==========================================
# 2. Preprocesare Augmentată
# ==========================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
# ==========================================
# 3. Pregătirea Datelor (Split Train/Test intern)
# ==========================================
# Încărcăm CSV-ul mare din folderul "labels"
df_mare = pd.read_csv("Dataset_KaggleEyePACS/labels/traintestLabels15_trainLabels19.csv")

# Împărțim cele 92.000 de imagini: 90% Antrenare, 10% Validare/Test
train_df, val_df = train_test_split(df_mare, test_size=0.10, random_state=42, stratify=df_mare['level'])

print(f"S-au alocat {len(train_df)} imagini pentru antrenare și {len(val_df)} pentru test/loss.")

dir_imagini_mare = "Dataset_KaggleEyePACS/resized_traintest15_train19"

train_dataset = CombinatEyePACSDataset(dataframe=train_df, img_dir=dir_imagini_mare, transform=train_transforms)
val_dataset = CombinatEyePACSDataset(dataframe=val_df, img_dir=dir_imagini_mare, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

S-au alocat 83127 imagini pentru antrenare și 9237 pentru test/loss.


In [6]:
# ==========================================
# 4. Definirea Modelului (ResNet50 Binar)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 1)
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [7]:
# ==========================================
# 5. Bucla de Antrenare
# ==========================================
num_epochs = 10 # Poți mări la 15 dacă ai timp; 10 e suficient pentru 80k poze
istoric_train_loss, istoric_val_loss = [], []
best_val_loss = float('inf')
cale_model_salvat = "cel_mai_bun_model_EyePACS_2015_2019.pth"

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
        
    avg_train_loss = running_train_loss / len(train_loader)
    istoric_train_loss.append(avg_train_loss)
    
    # Validare internă
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            loss = criterion(model(images), labels)
            running_val_loss += loss.item()
            
    avg_val_loss = running_val_loss / len(val_loader)
    istoric_val_loss.append(avg_val_loss)
    
    print(f"Epoca [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), cale_model_salvat)
        print("  -> Model Checkpoint salvat!")

Epoca [1/10] | Train Loss: 0.5550 | Val Loss: 0.5117
  -> Model Checkpoint salvat!


KeyboardInterrupt: 

In [ ]:
# ==========================================
# 6. Salvarea Graficului
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), istoric_train_loss, label='Train Loss', color='blue')
plt.plot(range(1, num_epochs + 1), istoric_val_loss, label='Validation Loss', color='red', linestyle='--')
plt.title('Evoluția Funcției de Cost (Set Masiv EyePACS 15+19)')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('grafic_eyepacs_combinat.png', dpi=300)
plt.show()

In [ ]:
import os
import pandas as pd
import torch
from torchvision import transforms, models
from torch import nn
from PIL import Image

def ruleaza_inferenta_oarba(cale_model_salvat, test_csv_path, test_images_dir, output_csv="predictii_test_19.csv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Încărcare model optimizat
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 1))
    model.load_state_dict(torch.load(cale_model_salvat, map_location=device))
    model = model.to(device)
    model.eval()

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 2. Încărcare CSV oarbă (Kaggle Test)
    test_df = pd.read_csv(test_csv_path)
    
    # Coloana se numește id_code în acest CSV
    nume_imagini = test_df['id_code'].tolist()
    
    predictii = []
    probabilitati = []

    print(f"Începem inferența pe {len(nume_imagini)} imagini...")

    with torch.no_grad():
        for nume in nume_imagini:
            # Găsim extensia reală
            img_path = os.path.join(test_images_dir, f"{nume}.jpeg")
            if not os.path.exists(img_path):
                img_path = os.path.join(test_images_dir, f"{nume}.png")
            
            try:
                image = Image.open(img_path).convert('RGB')
                image_tensor = val_transforms(image).unsqueeze(0).to(device)
                
                output = model(image_tensor)
                prob = torch.sigmoid(output).item()
                
                predictii.append(1 if prob >= 0.5 else 0)
                probabilitati.append(f"{prob*100:.2f}%")
            except Exception as e:
                print(f"Nu am putut citi {nume}: {e}")
                predictii.append(-1) # Marcăm erorile
                probabilitati.append("Eroare")

    # 3. Exportăm rezultatele finale
    test_df['Diagnostic_Retea'] = predictii
    test_df['Probabilitate_Bolnav'] = probabilitati
    
    test_df.to_csv(output_csv, index=False)
    print(f"Gata! Rezultatele au fost salvate în {output_csv}.")

# ==========================================
# Rularea inferenței:
# ==========================================
ruleaza_inferenta_oarba(
   "cel_mai_bun_model_EyePACS_2015_2019.pth", 
   "labels/testImages19.csv", 
   "resized_test19"
)